### 1. 消息概述

消息（Message）是 LangChain 里"一次对话的一轮发言"。无论是你问模型、模型回你，还是设定人设，本质都是一条消息。

每条消息都带一个 role（角色），用来标明"这句话是谁说的"：
- system —— 系统设定，用来定人设 / 定规则
- user —— 用户（人类）说的话
- assistant —— 模型（AI）的回复
- tool —— 工具调用后返回的结果（05 阶段详解）

为什么要区分 role？因为模型是"无状态"的——它不会记住上一轮。所以我们每次都要把整段对话（一串带 role 的消息）一起传给模型，模型靠 role 判断"哪些是背景、哪些是问题、哪些是我之前说过的话"。

### 2. 四种消息类型

In [1]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# ⭐ 四种消息，各司其职
system = SystemMessage(content="你是一个耐心的 Python 老师")
human = HumanMessage(content="什么是装饰器？")
ai = AIMessage(content="装饰器是一个接收函数、返回新函数的函数……")
tool = ToolMessage(content="查询结果：25°C", tool_call_id="call_001")

# 观察每种消息的类型和 content
for msg in [system, human, ai, tool]:
    print(type(msg).__name__, "=>", msg.content)

SystemMessage => 你是一个耐心的 Python 老师
HumanMessage => 什么是装饰器？
AIMessage => 装饰器是一个接收函数、返回新函数的函数……
ToolMessage => 查询结果：25°C


| 消息类型 | role | 谁说的 | 用途 |
|---------|------|--------|------|
| SystemMessage | system | 开发者 | 定人设、定规则（语气、格式、角色）|
| HumanMessage | user | 用户 | 提问、下指令 |
| AIMessage | assistant | 模型 | 模型的回答 |
| ToolMessage | tool | 工具 | 工具执行结果（Agent 阶段用）|

两种写法等价：字典格式 ↔ Message 对象（推荐用对象，可读性更好、功能更全）

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

model = init_chat_model(model="deepseek-flash")

# 写法一：字典格式
conversation_dict = [
    {"role": "system", "content": "你是翻译助手"},
    {"role": "user", "content": "翻译：我喜欢编程"},
]

# 写法二：Message 对象（推荐）
conversation_obj = [
    SystemMessage(content="你是翻译助手"),
    HumanMessage(content="翻译：我喜欢编程"),
]

# 两种写法效果完全一样
print("字典格式：", model.invoke(conversation_dict).content)
print("对象格式：", model.invoke(conversation_obj).content)

### 3. 消息的内部结构

一条 AIMessage 不止有 content，还携带很多元信息。用 model_dump_json() 可以看到完整字段。

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="deepseek-flash")
response = model.invoke("你好")

# ⭐ 查看 AIMessage 的完整字段
print(response.model_dump_json(indent=2))

```
AIMessage
├── content              最终文本
├── additional_kwargs    附加信息（如推理内容 reasoning_content）
├── response_metadata    模型原始元数据（token 用量、模型名）
├── id                   消息唯一 ID
├── tool_calls           工具调用（Agent 阶段用）
└── usage_metadata       标准化 token 用量
```

In [ ]:
# ⭐ 逐字段访问
print("正文：", response.content)
print("推理内容：", response.additional_kwargs.get("reasoning_content"))
print("token 用量：", response.usage_metadata)
print("模型元数据：", response.response_metadata)

### 4. 构建完整的多轮对话

把多轮消息按顺序放进列表，就是一次完整对话。模型会根据 role 理解上下文。

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = init_chat_model(model="deepseek-flash")

# ⭐ 多轮对话 = 按顺序的消息列表
conversation = [
    SystemMessage(content="你是一个记性很好的助手"),
    HumanMessage(content="我叫小明"),
    AIMessage(content="你好小明！"),
    HumanMessage(content="我叫什么名字？"),
]

response = model.invoke(conversation)
print(response.content)  # 模型能"记得"前面说过的名字

### 常见坑
- 导入路径用 `langchain_core.messages`（不是 `langchain.messages`）
- 新版本 content 支持"内容块列表"（content_blocks），可文本+图片混排，但纯文本时 content 仍是字符串
- 忘传历史消息 = 模型"失忆"，这正是 03-记忆与历史 要解决的问题